# Day 10 Content

In [1]:
import asyncio
import httpx
import time
import os

from json5 import host

BASE = 'https://dummyjson.com'
print('Ready')

Ready


In [3]:
start = time.perf_counter()
titles = []
for pid in [1,2,3,4,5]:
    r = httpx.get(f'{BASE}/products/{pid}', params={'delay': 500}, timeout=10.0)
    titles.append(r.json()['title'])
print("Got:", titles)
print(f'One after another took {time.perf_counter() - start:.1f}s (~5 x 0.5s = ~2.5s)')

Got: ['Essence Mascara Lash Princess', 'Eyeshadow Palette with Mirror', 'Powder Canister', 'Red Lipstick', 'Red Nail Polish']
One after another took 16.7s (~5 x 0.5s = ~2.5s)


In [5]:
async def get_one_title(pid):
    async with httpx.AsyncClient(base_url=BASE, timeout=15.0) as client:
        r = await client.get(f'/products/{pid}')
        return r.json()['title']

title = await get_one_title(1)
print('Product 1 is:', title)

Product 1 is: Essence Mascara Lash Princess


In [7]:
async def get_titles(pids):
    async with httpx.AsyncClient(base_url=BASE, timeout=15.0) as client:
        async def one(pid):
            r = await client.get(f'products/{pid}', params={'delay':500})
            return r.json()['title']

        jobs = [one(pid) for pid in pids]

        return await asyncio.gather(*jobs)

start = time.perf_counter()
titles = await get_titles([1,2,3,4,5])
print('Got:', titles)
print(f'All at once took  {time.perf_counter() - start:.1f}s (~0.5s, NOT ~2.5s!)')

Got: ['Essence Mascara Lash Princess', 'Eyeshadow Palette with Mirror', 'Powder Canister', 'Red Lipstick', 'Red Nail Polish']
All at once took  6.8s (~0.5s, NOT ~2.5s!)


In [8]:
# task definition
async def one(client, pid):
    r = await client.get(f'products/{pid}', params={'delay': 500})
    return r.json()['title']

async def get_titles(pids):
    client = httpx.AsyncClient(base_url=BASE, timeout=15.0)
    jobs = [one(client, pid) for pid in pids]
    results = await asyncio.gather(*jobs, return_exceptions=True)
    await client.aclose()
    return results

titles = await get_titles([1,2,3,99999,4,5])
print('Got:', titles)

Got: ['Essence Mascara Lash Princess', 'Eyeshadow Palette with Mirror', 'Powder Canister', KeyError('title'), ReadTimeout(''), 'Red Nail Polish']


In [9]:
async def get_or_error(pids):
    async with httpx.AsyncClient(base_url=BASE, timeout=15.0) as client:
        async def one(pid):
            r = await client.get(f'/products/{pid}')
            r.raise_for_status()
            return r.json()['title']

        return await asyncio.gather(
            *[one(pid) for pid in pids],
            return_exceptions=True,
        )

results = await get_or_error([1,2,99999,4])
for pid, res, in zip([1,2,99999,4], results):
    if isinstance(res, Exception):
        print(f'    product {pid}: FAILED ({type(res).__name__})')
    else:
        print(f'    product {pid}: {res}')

    product 1: Essence Mascara Lash Princess
    product 2: Eyeshadow Palette with Mirror
    product 99999: FAILED (HTTPStatusError)
    product 4: Red Lipstick


In [11]:
import os

try:
    import asyncpg
    from dotenv import load_dotenv
    load_dotenv()
    HAVE_PG = all(os.environ.get(k) for k in ['DB_NAME', 'DB_USER'])
except Exception:
    HAVE_PG = False

print('Postgres Available:', HAVE_PG)

Postgres Available: False


In [12]:
_pg_pool = None

async def get_pool():
    global _pg_pool
    if _pg_pool is None:
        _pg_pool = await asyncpg.create_pool(
            host     = os.environ.get('DB_HOST','localhost'),
            port     = int(os.environ.get('DB_PORT','5432')),
            database = os.environ.get('DB_NAME'),
            user     = os.environ.get('DB_USER'),
            password = os.environ.get('DB_PASSWORD'),
            min_size = 1,
            max_size = 5,
        )
    return _pg_pool

async def fetch_customer(customer_id):
    pool = await get_pool()
    async with pool.acquire(timeout=10) as conn:
        row = await conn.fetchrow(
            'SELECT * FROM customers WHERE customer_id = $1', customer_id)
        return dict(row) if row else None

if HAVE_PG:
    one = await fetch_customer(1001)
    print('Fetche (async):', one)
else:
    print('(skipped - no Postgres connected)')

(skipped - no Postgres connected)


In [13]:
if HAVE_PG:
    start = time.perf_counter()
    customers = await asyncio.gather(
        fetch_customer(1001),
        fetch_customer(1002),
        fetch_customer(1003),
    )

    for c in customers:
        print('   ', c)
    print(f'Three real DB lookups together took {time.perf_counter() - start:.2f}s')
else:
    print('(skipped - no postgres connected)')

(skipped - no postgres connected)
